# Practical worksheet: Classical Generative AI with L-systems and IFS

This notebook is a **guided implementation**.
You will code classical generators with explicit rules and stochastic transitions:

1. Deterministic L-systems (production rules)
2. Stochastic L-systems (Markov-style local choices)
3. IFS (Iterated Function Systems)

Important: in this student version, many blocks are intentionally incomplete.
Look for `TODO START` / `TODO END` markers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1) Tiny demo first: how rewriting rules work

Before building the full algorithm, run a minimal symbolic example.

Rule set:
- `A -> AB`
- `B -> A`

Start from `A` and apply rules for a few iterations.


In [1]:
# Simple complete demo (already solved)
demo_rules = {
    "A": "AB",
    "B": "A",
}

sentence = "A"
for step in range(6):
    print(f"step={step:>2} | len={len(sentence):>3} | {sentence}")
    sentence = "".join(demo_rules.get(ch, ch) for ch in sentence)

step= 0 | len=  1 | A
step= 1 | len=  2 | AB
step= 2 | len=  3 | ABA
step= 3 | len=  5 | ABAAB
step= 4 | len=  8 | ABAABABA
step= 5 | len= 13 | ABAABABAABAAB


### 1.1 Your implementation: generic deterministic L-system

Pseudo-code:

```text
function rewrite_once(sentence, rules):
    out <- empty list
    for each symbol s in sentence:
        if s in rules: append rules[s]
        else: append s
    return join(out)

function generate_lsystem(axiom, rules, n_iter):
    sentence <- axiom
    repeat n_iter times:
        sentence <- rewrite_once(sentence, rules)
    return sentence
```

Hint: use `rules.get(symbol, symbol)`.


In [ ]:
def rewrite_once(sentence, rules):
    # TODO START
    # Build a new list of symbols by replacing each symbol with its rule (if any).
    # Return a single joined string.
    raise NotImplementedError("Implement rewrite_once")
    # TODO END


def generate_lsystem(axiom, rules, n_iter):
    # TODO START
    # Repeatedly call rewrite_once for n_iter steps.
    raise NotImplementedError("Implement generate_lsystem")
    # TODO END


### 1.2 Small starter rule set (do this first)

Use this small system before the big branching one:

- axiom: `F`
- rule: `F -> F+F-F-F+F`

Tasks:
1. Choose `n_iter_small` (start with 2 or 3).
2. Generate `sentence_small`.
3. Print its length.


In [ ]:
axiom_small = "F"
rules_small = {
    "F": "F+F-F-F+F"
}

# TODO START
n_iter_small = None
sentence_small = None
# TODO END

print("Small sentence length:", len(sentence_small))
print(sentence_small[:120] + "...")


## 2) Turtle rendering (geometry layer)

Interpretation:
- `F`: draw forward
- `+`: left turn by `angle_deg`
- `-`: right turn by `angle_deg`
- `[` push state `(x, y, heading)`
- `]` pop state

Pseudo-code:

```text
for command in commands:
    if command == F: draw segment and update position
    elif command == +: heading += angle
    elif command == -: heading -= angle
    elif command == [: push current state
    elif command == ]: pop state if stack not empty
```


In [ ]:
def turtle_trace(commands, angle_deg=90.0, step=5.0):
    x, y = 0.0, 0.0
    heading = 0.0
    stack = []

    xs, ys = [], []

    def draw_forward(x, y, heading, step):
        rad = np.deg2rad(heading)
        xn = x + step * np.cos(rad)
        yn = y + step * np.sin(rad)
        return xn, yn

    for c in commands:
        if c == "F":
            xn, yn = draw_forward(x, y, heading, step)
            xs.extend([x, xn, np.nan])
            ys.extend([y, yn, np.nan])
            x, y = xn, yn
        elif c == "+":
            heading = heading + angle_deg
        elif c == "-":
            heading = heading - angle_deg
        elif c == "[":
            stack.append((x, y, heading))
        elif c == "]" and len(stack) > 0:
            x, y, heading = stack.pop()

    return np.array(xs), np.array(ys)


In [ ]:
# Visualize the small system first
xs_small, ys_small = turtle_trace(sentence_small, angle_deg=90.0, step=2.0)

plt.figure(figsize=(7, 4))
plt.plot(xs_small, ys_small, color="steelblue", lw=0.8)
plt.axis("equal")
plt.axis("off")
plt.title("Small deterministic L-system")
plt.show()


### 2.1 Bigger deterministic system (main challenge)

Now move to a branching plant-like grammar:

- axiom: `X`
- rules:
  - `X -> F+[[X]-X]-F[-FX]+X`
  - `F -> FF`

Tasks:
1. Pick `n_iter_big` (try 4 or 5 first).
2. Pick `angle_big` (around 20 to 30 degrees usually works).
3. Pick `step_big` and render.


In [ ]:
axiom_big = "X"
rules_big = {
    "X": "F+[[X]-X]-F[-FX]+X",
    "F": "FF"
}

# TODO START
n_iter_big = None
angle_big = None
step_big = None
sentence_big = None
# TODO END

print("Big sentence length:", len(sentence_big))

xs_big, ys_big = turtle_trace(sentence_big, angle_deg=angle_big, step=step_big)
plt.figure(figsize=(7, 7))
plt.plot(xs_big, ys_big, color="forestgreen", lw=0.7)
plt.axis("equal")
plt.axis("off")
plt.title("Bigger deterministic L-system")
plt.show()


## 3) Stochastic L-system (Markov-style local transitions)

For each symbol, choose one rewrite by probability.

Pseudo-code:

```text
sample_rule(symbol):
    draw u in [0,1)
    cumulative <- 0
    for (replacement, prob) in choices:
        cumulative += prob
        if u <= cumulative: return replacement
    return last replacement

rewrite_stochastic(sentence):
    for each symbol:
        if symbol has stochastic rules: sample replacement
        else: keep symbol
```

Hint: probabilities should sum to approximately 1.


In [ ]:
def sample_rule(symbol, stochastic_rules, rng):
    # TODO START
    # Implement cumulative-probability sampling.
    raise NotImplementedError("Implement sample_rule")
    # TODO END


def rewrite_stochastic(sentence, stochastic_rules, rng):
    # TODO START
    # Build new sentence symbol by symbol; call sample_rule where needed.
    raise NotImplementedError("Implement rewrite_stochastic")
    # TODO END


def generate_stochastic(axiom, stochastic_rules, n_iter, seed=0):
    # TODO START
    # Initialize rng and iteratively rewrite n_iter times.
    raise NotImplementedError("Implement generate_stochastic")
    # TODO END


### Quick check (expected)

- Different seeds should lead to different structures.
- If probabilities do not sum near 1.0, behavior is unstable.


In [ ]:
stochastic_rules = {
    "F": [
        ("F[+F]F[-F]F", 0.34),
        ("F[+F]F", 0.33),
        ("F[-F]F", 0.33),
    ]
}

# TODO START
seed = None
sentence_sto = None
# TODO END

print("Stochastic sentence length:", len(sentence_sto))

xs_s, ys_s = turtle_trace(sentence_sto, angle_deg=22.5, step=2.0)
plt.figure(figsize=(7, 7))
plt.plot(xs_s, ys_s, color="darkolivegreen", lw=0.8)
plt.axis("equal")
plt.axis("off")
plt.title("Stochastic L-system")
plt.show()


## 4) IFS (Iterated Function Systems)

**Simple idea:**
An IFS creates a shape by repeatedly applying one affine transformation selected at random.
At each step, you pick one rule `i` with probability `p_i` and update:

`x_{t+1} = A_i x_t + b_i`

After many iterations, points tend to concentrate on a fractal attractor (e.g., Barnsley fern).

### Why burn-in?
We start from an arbitrary initial point (usually `[0,0]`).
The first iterations are strongly influenced by this initial condition (transient phase),
so we discard them using `burnin`.

- `burnin` = number of initial points ignored
- Purpose: keep mostly points from the stable attractor distribution

Pseudo-code:

```text
x <- [0, 0]
points <- []
repeat n_points + burnin times:
    sample index i with probabilities p
    x <- A_i @ x + b_i
    if after burnin: append copy of x
return points
```


In [ ]:
def ifs_sample(transformations, probs, n_points=50000, burnin=100, seed=0):
    rng = np.random.default_rng(seed)

    # TODO START
    # 1) Initialize current point x (2D) and an empty list pts.
    # 2) Loop for n_points + burnin steps.
    # 3) Sample transform index i with rng.choice(..., p=probs).
    # 4) Update x = A @ x + b.
    # 5) After burn-in, store x.copy().
    # 6) Return np.array(pts).
    raise NotImplementedError("Implement ifs_sample")
    # TODO END


In [ ]:
# Barnsley fern parameters
transforms = [
    (np.array([[0.00, 0.00], [0.00, 0.16]]), np.array([0.0, 0.0])),
    (np.array([[0.85, 0.04], [-0.04, 0.85]]), np.array([0.0, 1.6])),
    (np.array([[0.20, -0.26], [0.23, 0.22]]), np.array([0.0, 1.6])),
    (np.array([[-0.15, 0.28], [0.26, 0.24]]), np.array([0.0, 0.44])),
]
probs = [0.01, 0.85, 0.07, 0.07]

# TODO START
n_points = None
pts = None
# TODO END

plt.figure(figsize=(6, 10))
plt.scatter(pts[:, 0], pts[:, 1], s=0.08, color="seagreen")
plt.axis("equal")
plt.axis("off")
plt.title("IFS: Barnsley fern")
plt.show()


## 5) Exploration Assignment: experiment + implement

Goal: run controlled experiments on the three generators and document what changes and why.

### Task A: Deterministic L-system scaling

1. Run the **small rule set** with `n_iter_small in {1,2,3,4}`.
2. For each run, record:
- sentence length
- visual complexity (your own qualitative note)
3. Keep angle fixed, then change angle and compare geometry.

Implementation suggestions:
- Create a loop over parameter values.
- Store results in a list of dicts: `{"n_iter": ..., "len": ...}`.
- Print a compact table before plotting.

### Task B: Stochastic diversity

1. Keep same rules, vary `seed` (at least 5 values).
2. Measure diversity with simple proxies:
- final sentence length
- number of branch tokens: `sentence.count('[')`
3. Optionally change probability vector and compare output variety.

Implementation suggestions:
- Write `analyze_sentence(sentence)` returning a small metrics dict.
- Use one summary plot (e.g., bar plot of branch counts by seed).

### Task C: IFS convergence and burn-in

1. Test `burnin in {0, 20, 100, 500}` with fixed `n_points`.
2. Test `n_points in {2000, 10000, 50000}` with fixed `burnin`.
3. Compare visual stability and noise.

Implementation suggestions:
- Use subplots to compare settings side by side.
- Keep same random seed for fair comparison.
- Briefly explain when burn-in is most important.


## 6) Theoretical questions (short answers)

1. Why is deterministic rewriting a production-system style method?
2. Where exactly is the Markov-style step in stochastic L-systems?
3. Why is IFS generative even without neural networks?


Answers:

1.

2.

3.

